In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
MODELS.mkdir(exist_ok=True)

In [3]:
chunks = pd.read_json(PROCESSED / "chunks.jsonl", lines=True)
required = ["chunk_id", "bank", "bank_code", "document_type", "fiscal_year", "quarter",
            "section", "risk_category", "page_start", "page_end", "text"]
missing = [c for c in required if c not in chunks.columns]
assert not missing, f"Colonnes manquantes : {missing}. Relance le notebook 01."
assert chunks.chunk_id.is_unique

print(f"{len(chunks)} chunks chargés")
print(chunks.groupby(["bank_code", "fiscal_year"]).size().unstack())
print("\nValeurs manquantes par colonne de métadonnées :")
print(chunks[required[:-1]].isna().sum().to_string())

3562 chunks chargés
fiscal_year  2024  2025
bank_code              
BMO           339   314
BNS           359   358
CIBC          354   332
RBC           365   328
TD            412   401

Valeurs manquantes par colonne de métadonnées :
chunk_id         0
bank             0
bank_code        0
document_type    0
fiscal_year      0
quarter          0
section          0
risk_category    0
page_start       0
page_end         0


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print("Limite par défaut :", model.max_seq_length, "tokens")
print("Limite physique du modèle :", model[0].auto_model.config.max_position_embeddings, "tokens")

model.max_seq_length = 512          # le maximum possible pour ce modèle

ids = model.tokenizer(chunks.text.tolist(), add_special_tokens=True, truncation=False)["input_ids"]
chunks["n_tokens_minilm"] = [len(x) for x in ids]

print("\nTaille des chunks avec le tokenizer de MiniLM :")
print(chunks.n_tokens_minilm.describe().round(0).to_string())
for limit in (256, 512):
    print(f"Chunks dépassant {limit} tokens (donc tronqués) : {(chunks.n_tokens_minilm > limit).mean():.0%}")
seen = np.minimum(chunks.n_tokens_minilm, 512).sum() / chunks.n_tokens_minilm.sum()
print(f"Part du texte vue par MiniLM avec la limite de 512 : {seen:.0%}")

/Users/carchaf/jarvis_data_eng_SamiaCarchaf/module4_genai_bank_rag/.venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/Users/carchaf/jarvis_data_eng_SamiaCarchaf/module4_genai_bank_rag/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Limite par défaut : 256 tokens
Limite physique du modèle : 512 tokens


Token indices sequence length is longer than the specified maximum sequence length for this model (864 > 256). Running this sequence through the model will result in indexing errors



Taille des chunks avec le tokenizer de MiniLM :
count    3562.0
mean      724.0
std       160.0
min        79.0
25%       633.0
50%       739.0
75%       843.0
max      1140.0
Chunks dépassant 256 tokens (donc tronqués) : 99%
Chunks dépassant 512 tokens (donc tronqués) : 90%
Part du texte vue par MiniLM avec la limite de 512 : 69%


In [ ]:
model.max_seq_length = 512            # maximum de MiniLM

t0 = time.time()
emb = model.encode(
    chunks.text.tolist(),
    batch_size=32,
    normalize_embeddings=True,        # vecteurs de norme 1 : cosinus = produit scalaire
    show_progress_bar=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0

print(f"\n{emb.shape[0]} embeddings de dimension {emb.shape[1]} en {elapsed/60:.1f} min "
      f"({emb.shape[0]/elapsed:.1f} chunks/s) | coût : $0 (local)")
norms = np.linalg.norm(emb, axis=1)
print(f"Normes : min {norms.min():.4f}, max {norms.max():.4f} | valeurs NaN : {int(np.isnan(emb).sum())}")

# Sauvegarde : on ne recalculera pas ça à chaque fois (dossier models/, ignoré par Git)
np.save(MODELS / "minilm_embeddings.npy", emb)
chunks[["chunk_id"]].to_csv(MODELS / "minilm_chunk_ids.csv", index=False)
with open(MODELS / "minilm_config.json", "w") as f:
    json.dump({
        "model": "all-MiniLM-L6-v2", "dim": int(emb.shape[1]), "max_seq_length": 512, "normalized": True,
        "n_chunks": int(emb.shape[0]), "seconds": round(elapsed, 1),
        "share_chunks_truncated_512": round(float((chunks.n_tokens_minilm > 512).mean()), 3),
        "share_text_seen_512": round(float(seen), 3),
    }, f, indent=2)
print("Sauvegardé dans models/ (embeddings, ids, configuration)")

Batches: 100%|██████████| 112/112 [06:46<00:00,  3.63s/it]


3562 embeddings de dimension 384 en 6.8 min (8.7 chunks/s) | coût : $0 (local)
Normes : min 1.0000, max 1.0000 | valeurs NaN : 0
Sauvegardé dans models/ (embeddings, ids, configuration)


In [6]:
import chromadb
from chromadb.config import Settings

COLLECTION = "bank_annual_reports_minilm"
client = chromadb.PersistentClient(path=str(MODELS / "chroma"), settings=Settings(anonymized_telemetry=False))

if COLLECTION in [c.name for c in client.list_collections()]:      # on repart d'une collection vide à chaque exécution
    client.delete_collection(COLLECTION)
col = client.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})

meta_cols = ["bank", "bank_code", "document_type", "fiscal_year", "quarter", "part", "section",
             "risk_category", "page_start", "page_end", "has_table", "n_tokens", "citation"]
metadatas = []
for r in chunks[meta_cols].to_dict("records"):
    metadatas.append({k: (bool(v) if k == "has_table" else int(v) if k in ("fiscal_year", "page_start", "page_end", "n_tokens") else str(v))
                      for k, v in r.items()})

t0 = time.time()
BATCH = 500
for i in range(0, len(chunks), BATCH):
    sl = slice(i, i + BATCH)
    col.add(ids=chunks.chunk_id.tolist()[sl], embeddings=emb[sl].tolist(),
            documents=chunks.text.tolist()[sl], metadatas=metadatas[sl])
print(f"{col.count()} chunks insérés en {time.time() - t0:.0f} s")

# Contrôles
assert col.count() == len(chunks), "Le nombre de chunks insérés ne correspond pas"
print("Collection rouverte depuis le disque :",
      chromadb.PersistentClient(path=str(MODELS / "chroma"), settings=Settings(anonymized_telemetry=False))
      .get_collection(COLLECTION).count(), "chunks")

# Test : un chunk doit être son propre plus proche voisin (distance ~ 0)
probe = chunks.iloc[1234]
res = col.query(query_embeddings=[emb[1234].tolist()], n_results=3, include=["distances", "metadatas"])
print(f"\nTest : chunk {probe.chunk_id}")
for cid, d, m in zip(res["ids"][0], res["distances"][0], res["metadatas"][0]):
    print(f"  {cid}  distance={d:.4f}  similarité={1 - d:.3f}  {m['citation'][:70]}")

# Test des filtres : combien de chunks par banque et par année dans la collection ?
for b in ["RBC", "TD", "BMO", "BNS", "CIBC"]:
    n = {y: len(col.get(where={"$and": [{"bank_code": b}, {"fiscal_year": y}]}, include=[])["ids"]) for y in (2024, 2025)}
    print(f"  {b:5} {n}")

3562 chunks insérés en 7 s
Collection rouverte depuis le disque : 3562 chunks

Test : chunk BNS_annual_report_FY2025-0222
  BNS_annual_report_FY2025-0222  distance=0.0000  similarité=1.000  Bank of Nova Scotia – Annual Report FY2025 – Consolidated Statement Of
  CIBC_annual_report_FY2025-0186  distance=0.1226  similarité=0.877  Canadian Imperial Bank of Commerce – Annual Report FY2025 – Consolidat
  BNS_annual_report_FY2024-0220  distance=0.1281  similarité=0.872  Bank of Nova Scotia – Annual Report FY2024 – Consolidated Statement Of
  RBC   {2024: 365, 2025: 328}
  TD    {2024: 412, 2025: 401}
  BMO   {2024: 339, 2025: 314}
  BNS   {2024: 359, 2025: 358}
  CIBC  {2024: 354, 2025: 332}


In [7]:
SIM_MIN, LAMBDA, K, FETCH_K = 0.3, 0.7, 5, 20
REPORTS = ROOT / "reports"
RETRIEVAL_LOG = []

def build_where(bank=None, year=None):
    conds = []
    if bank: conds.append({"bank_code": bank})
    if year: conds.append({"fiscal_year": year})
    if not conds: return None
    return conds[0] if len(conds) == 1 else {"$and": conds}

def embed_query(q):                      # même modèle qu'à l'indexation
    return model.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0]

def to_rows(res):
    return [{"chunk_id": cid, "sim": 1 - d, "citation": m["citation"], "bank_code": m["bank_code"],
             "fiscal_year": m["fiscal_year"], "risk_category": m["risk_category"], "text": doc}
            for cid, doc, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]

def retrieve_topk(q, k=K, where=None):
    res = col.query(query_embeddings=[embed_query(q).tolist()], n_results=k, where=where,
                    include=["documents", "metadatas", "distances"])
    return to_rows(res)

def retrieve_mmr(q, k=K, fetch_k=FETCH_K, lam=LAMBDA, where=None):
    qv = embed_query(q)
    res = col.query(query_embeddings=[qv.tolist()], n_results=fetch_k, where=where,
                    include=["documents", "metadatas", "distances", "embeddings"])
    rows = to_rows(res)
    cand = np.array(res["embeddings"][0])
    rel = cand @ qv                                   # vecteurs normalisés : produit scalaire = cosinus
    selected, remaining = [], list(range(len(rows)))
    while remaining and len(selected) < k:
        if not selected:
            best = max(remaining, key=lambda i: rel[i])
        else:
            sel = cand[selected]
            best = max(remaining, key=lambda i: lam * rel[i] - (1 - lam) * float(np.max(sel @ cand[i])))
        selected.append(best); remaining.remove(best)
    return [rows[i] for i in selected]

def search(qid, q, method, where=None):
    results = (retrieve_topk if method == "topk" else retrieve_mmr)(q, where=where)
    for rank, r in enumerate(results, 1):
        RETRIEVAL_LOG.append({"qid": qid, "method": method, "rank": rank, "chunk_id": r["chunk_id"],
                              "sim": round(r["sim"], 4), "kept": bool(r["sim"] >= SIM_MIN),
                              "where": str(where), "query": q})
    kept = [r for r in results if r["sim"] >= SIM_MIN]
    if not kept:
        print(f"  [{qid}/{method}] No relevant information found (aucun chunk au-dessus de {SIM_MIN})")
    return kept

QUERIES = [
    dict(id=1,  q="What was RBC's provision for credit losses in fiscal 2025?",                              type="factual", bank="RBC",  year=2025, kw=r"provision for credit losses|\bPCL\b"),
    dict(id=2,  q="What was TD's Liquidity Coverage Ratio as at October 31, 2025?",                           type="factual", bank="TD",   year=2025, kw=r"liquidity coverage ratio|\bLCR\b"),
    dict(id=3,  q="How does BMO describe its approach to climate risk management?",                          type="broad",   bank="BMO",  year=2025, kw=r"climate"),
    dict(id=4,  q="What does Scotiabank say about cybersecurity and technology risk?",                        type="broad",   bank="BNS",  year=2025, kw=r"cyber|technology risk"),
    dict(id=5,  q="What was CIBC's CET1 capital ratio at October 31, 2024?",                                  type="factual", bank="CIBC", year=2024, kw=r"\bCET1\b"),
    dict(id=6,  q="Which banks discuss artificial intelligence or OSFI Guideline E-23 on model risk?",        type="broad",   bank=None,   year=None, kw=r"artificial intelligence|E-23|model risk"),
    dict(id=7,  q="What were RBC's gross impaired loans in fiscal 2024?",                                     type="factual", bank="RBC",  year=2024, kw=r"gross impaired loans|impaired loans"),
    dict(id=8,  q="How sensitive are TD's net interest income and economic value of equity to interest rate changes?", type="factual", bank="TD", year=2024, kw=r"economic value of equity|\bEVE\b"),
    dict(id=9,  q="How is the credit quality of BMO's residential mortgage portfolio?",                       type="broad",   bank="BMO",  year=2024, kw=r"mortgage"),
    dict(id=10, q="What emerging risks does Scotiabank identify for 2024?",                                   type="broad",   bank="BNS",  year=2024, kw=r"emerging risk"),
]

results, rows = {}, []
for Q in QUERIES:
    where = build_where(Q["bank"], Q["year"])
    for method in ("topk", "mmr"):
        res = search(Q["id"], Q["q"], method, where)
        results[(Q["id"], method)] = res
        rows.append({"Q": Q["id"], "type": Q["type"], "filtre": f"{Q['bank'] or 'toutes'} {Q['year'] or ''}".strip(), "méthode": method,
                     "résultats": len(res), "sim max": round(max((r["sim"] for r in res), default=0), 3),
                     "sim moy": round(float(np.mean([r["sim"] for r in res])) if res else 0, 3),
                     "banques": len({r["bank_code"] for r in res}), "sections": len({r["citation"].split(" – ")[2] for r in res})})
print(pd.DataFrame(rows).to_string(index=False))

for Q in QUERIES:
    a = {r["chunk_id"] for r in results[(Q["id"], "topk")]}
    b = {r["chunk_id"] for r in results[(Q["id"], "mmr")]}
    print(f"Q{Q['id']:>2} : {len(a & b)}/5 chunks en commun entre top-k et MMR")

pd.DataFrame(RETRIEVAL_LOG).to_json(REPORTS / "retrieval_log.jsonl", orient="records", lines=True, force_ascii=False)
print(f"\nJournal : {len(RETRIEVAL_LOG)} lignes -> reports/retrieval_log.jsonl")

print("\nTest du seuil de similarité (0,3) :")
for q in ["What is Desjardins' CET1 ratio?", "What is the capital of France?"]:
    r = retrieve_topk(q, k=3)
    print(f"  {q!r} -> sim max {r[0]['sim']:.3f} | au-dessus du seuil : {sum(x['sim'] >= SIM_MIN for x in r)}/3 | meilleur : {r[0]['citation'][:55]}")

 Q    type    filtre méthode  résultats  sim max  sim moy  banques  sections
 1 factual  RBC 2025    topk          5    0.520    0.476        1         5
 1 factual  RBC 2025     mmr          5    0.520    0.475        1         5
 2 factual   TD 2025    topk          5    0.450    0.413        1         5
 2 factual   TD 2025     mmr          5    0.450    0.410        1         5
 3   broad  BMO 2025    topk          5    0.577    0.559        1         5
 3   broad  BMO 2025     mmr          5    0.577    0.553        1         5
 4   broad  BNS 2025    topk          5    0.583    0.541        1         5
 4   broad  BNS 2025     mmr          5    0.583    0.536        1         5
 5 factual CIBC 2024    topk          5    0.512    0.503        1         5
 5 factual CIBC 2024     mmr          5    0.512    0.474        1         5
 6   broad    toutes    topk          5    0.604    0.573        2         4
 6   broad    toutes     mmr          5    0.604    0.557        3         5

In [9]:
import re

def snippet(text, kw, width=280):
    m = re.search(kw, text, re.I)
    s = max(0, m.start() - 100) if m else 0
    return " ".join(text[s:s + width].split())

def show(chunk_id):
    print(chunks.loc[chunks.chunk_id == chunk_id].iloc[0].text)

REVIEW = {}
for Q in QUERIES:
    tk = results[(Q["id"], "topk")][:3]
    mm = results[(Q["id"], "mmr")][:3]
    items = {}
    for method, lst in (("topk", tk), ("mmr", mm)):
        for rank, r in enumerate(lst, 1):
            it = items.setdefault(r["chunk_id"], {"r": r, "topk": None, "mmr": None})
            it[method] = rank
    print(f"\n=== Q{Q['id']} ({Q['type']}) | {Q['q']}")
    REVIEW[Q["id"]] = []
    for letter, (cid, it) in zip("ABCDEF", items.items()):
        r = it["r"]
        ok_bank = "n/a" if not Q["bank"] else ("✓" if r["bank_code"] == Q["bank"] else "✗")
        ok_year = "n/a" if not Q["year"] else ("✓" if r["fiscal_year"] == Q["year"] else "✗")
        ok_kw = "✓" if re.search(Q["kw"], r["text"], re.I) else "✗"
        print(f"  [{letter}] {cid} | sim {r['sim']:.2f} | top-k #{it['topk'] or '-'} | MMR #{it['mmr'] or '-'} | banque {ok_bank} année {ok_year} mot-clé {ok_kw}")
        print(f"      {r['citation']}")
        print(f"      …{snippet(r['text'], Q['kw'])}…")
        REVIEW[Q["id"]].append((letter, cid, it["topk"], it["mmr"]))


=== Q1 (factual) | What was RBC's provision for credit losses in fiscal 2025?
  [A] RBC_annual_report_FY2025-0235 | sim 0.52 | top-k #1 | MMR #1 | banque ✓ année ✓ mot-clé ✓
      Royal Bank of Canada – Annual Report FY2025 – Note 5 Loans and allowance for credit losses (p.160-161)
      …5 $ 126 $ 231 $ 572 $ 223 $ 90 $ 168 $ 481 Provision for credit losses Transfers to stage 1 157 (153) (4) – 99 (97) (2)…
  [B] RBC_annual_report_FY2025-0141 | sim 0.48 | top-k #2 | MMR #- | banque ✓ année ✓ mot-clé ✗
      Royal Bank of Canada – Annual Report FY2025 – Critical Accounting Policies And Estimates (p.100-101)
      …Valuation adjustments may be subjective as they require significant judgment in the input selection, such as implied PD and recovery rate, and are intended to arrive at a fair value that is determined based on assumptions that market participants would use in pricing the financia…
  [C] RBC_annual_report_FY2025-0151 | sim 0.47 | top-k #3 | MMR #- | banque ✓ année ✓ mot-clé ✓


### Observations

#### Embeddings
- Modèle : `all-MiniLM-L6-v2` (local, gratuit), vecteurs de 384 dimensions normalisés.
- 3 562 chunks embeddés en 6,8 minutes (8,7 chunks/s) sur CPU, coût 0 $.
- **Troncature :** MiniLM lit au maximum 512 tokens (256 par défaut). Avec ses propres tokens, nos chunks font 724 tokens en moyenne : **90 % dépassent 512**, et le modèle ne voit que **69 % du texte**. Le texte complet reste stocké et sera donné au LLM, mais le vecteur de recherche ne représente que le début de chaque chunk.
- Les embeddings OpenAI (limite de 8 191 tokens) ne tronquent pas : c'est l'enjeu de la comparaison de la consigne 8.

#### Base vectorielle (ChromaDB)
- Collection persistante, similarité cosinus, 3 562 chunks insérés. Les comptes par banque et par année correspondent à ceux du notebook 01.
- Métadonnées : banque, type de document, année fiscale, trimestre, partie, section, catégorie de risque, pages, citation.
- Le filtrage par banque et année fonctionne : toutes les réponses des requêtes filtrées viennent de la bonne banque et de la bonne année.
- Sans filtre, une recherche peut renvoyer la mauvaise banque : le plus proche voisin d'un état financier de BNS était celui de CIBC (similarité 0,877).

#### Top-k contre MMR (k = 5, lambda = 0,7, 10 requêtes)
- MMR diversifie peu : 2 à 5 chunks sur 5 en commun avec top-k. Il est identique à top-k pour Q9 et remplace deux bons résultats par des résultats moins proches pour Q7.
- Sur la question sans filtre (Q6), MMR renvoie 3 banques contre 2 pour top-k.
- Le chevauchement entre chunks fait parfois remonter deux fois le même passage (Q9).

#### Seuil de similarité (0,3)
- Il écarte les questions hors domaine : « capitale de la France », similarité maximale de 0,185.
- Il **ne détecte pas une banque absente de la base** : « CET1 de Desjardins » atteint 0,408 et renvoie des chunks de BMO. Limite à traiter au notebook 04 (vérifier le nom de la banque avant la recherche).

#### Qualité du retrieval (PRÉLIMINAIRE, à confirmer par la vérification manuelle)
- Les questions descriptives avec un sujet ciblé fonctionnent bien (Q3, BMO et le climat : 3 résultats sur 3 pertinents).
- Certaines questions échouent : Q4 (cybersécurité BNS), Q8 (sensibilité aux taux de TD) et Q10 (risques émergents BNS) renvoient des pages de présentation ou des notes sans rapport.
- Les similarités sont modestes (0,41 à 0,68), ce qui est attendu pour une question courte face à un chunk de 800 tokens tronqué.
- Hypothèses à tester : (1) la troncature à 512 tokens, (2) le nom de la banque dans la requête, alors que les chunks parlent de « the Bank », (3) des titres de section imprécis.

#### Precision@3 (à compléter après la vérification manuelle)
| Méthode | Pertinents / 30 | Precision@3 |
|---|---|---|
| Top-k | ? | ? |
| MMR | ? | ? |## Observations : embeddings et recherche (étape 2)

#### Embeddings
- Modèle : `all-MiniLM-L6-v2` (local, gratuit), vecteurs de 384 dimensions normalisés.
- 3 562 chunks embeddés en 6,8 minutes (8,7 chunks/s) sur CPU, coût 0 $.
- **Troncature :** MiniLM lit au maximum 512 tokens (256 par défaut). Avec ses propres tokens, nos chunks font 724 tokens en moyenne : **90 % dépassent 512**, et le modèle ne voit que **69 % du texte**. Le texte complet reste stocké et sera donné au LLM, mais le vecteur de recherche ne représente que le début de chaque chunk.
- Les embeddings OpenAI (limite de 8 191 tokens) ne tronquent pas : c'est l'enjeu de la comparaison de la consigne 8.

#### Base vectorielle (ChromaDB)
- Collection persistante, similarité cosinus, 3 562 chunks insérés. Les comptes par banque et par année correspondent à ceux du notebook 01.
- Métadonnées : banque, type de document, année fiscale, trimestre, partie, section, catégorie de risque, pages, citation.
- Le filtrage par banque et année fonctionne : toutes les réponses des requêtes filtrées viennent de la bonne banque et de la bonne année.
- Sans filtre, une recherche peut renvoyer la mauvaise banque : le plus proche voisin d'un état financier de BNS était celui de CIBC (similarité 0,877).

#### Top-k contre MMR (k = 5, lambda = 0,7, 10 requêtes)
- MMR diversifie peu : 2 à 5 chunks sur 5 en commun avec top-k. Il est identique à top-k pour Q9 et remplace deux bons résultats par des résultats moins proches pour Q7.
- Sur la question sans filtre (Q6), MMR renvoie 3 banques contre 2 pour top-k.
- Le chevauchement entre chunks fait parfois remonter deux fois le même passage (Q9).

#### Seuil de similarité (0,3)
- Il écarte les questions hors domaine : « capitale de la France », similarité maximale de 0,185.
- Il **ne détecte pas une banque absente de la base** : « CET1 de Desjardins » atteint 0,408 et renvoie des chunks de BMO. Limite à traiter au notebook 04 (vérifier le nom de la banque avant la recherche).

#### Qualité du retrieval (PRÉLIMINAIRE, à confirmer par la vérification manuelle)
- Les questions descriptives avec un sujet ciblé fonctionnent bien (Q3, BMO et le climat : 3 résultats sur 3 pertinents).
- Certaines questions échouent : Q4 (cybersécurité BNS), Q8 (sensibilité aux taux de TD) et Q10 (risques émergents BNS) renvoient des pages de présentation ou des notes sans rapport.
- Les similarités sont modestes (0,41 à 0,68), ce qui est attendu pour une question courte face à un chunk de 800 tokens tronqué.
- Hypothèses à tester : (1) la troncature à 512 tokens, (2) le nom de la banque dans la requête, alors que les chunks parlent de « the Bank », (3) des titres de section imprécis.

#### Precision@3 (à compléter après la vérification manuelle)
| Méthode | Pertinents / 30 | Precision@3 |
|---|---|---|
| Top-k | ? | ? |
| MMR | ? | ? |

### vérifier les cas douteux

In [10]:
def window(chunk_id, kw, n=2, width=420):
    t = chunks.loc[chunks.chunk_id == chunk_id].iloc[0].text
    print(f"\n--- {chunk_id} ---")
    ms = list(re.finditer(kw, t, re.I))[:n]
    if not ms:
        print("   (mot-clé absent)"); return
    for m in ms:
        s = max(0, m.start() - 120)
        print("   …" + " ".join(t[s:s + width].split()) + "…")

CHECKS = [
    ("Q1-A : le total de la PCL 2025 y est-il ?",            "RBC_annual_report_FY2025-0235", r"provision for credit losses"),
    ("Q1-C : le total de la PCL 2025 y est-il ?",            "RBC_annual_report_FY2025-0151", r"provision for credit losses"),
    ("Q2-B : LCR au 31 octobre 2025 ?",                       "TD_annual_report_FY2025-0182",  r"\bLCR\b|liquidity coverage"),
    ("Q5-B : valeur du ratio CET1 au 31 oct. 2024 ?",        "CIBC_annual_report_FY2024-0075", r"CET1 ratio"),
    ("Q7-A : 5 867 = prêts douteux bruts ?",                  "RBC_annual_report_FY2024-0176", r"impaired loans"),
    ("Q7-B : montant des prêts douteux bruts 2024 ?",         "RBC_annual_report_FY2024-0102", r"gross impaired loans"),
    ("Q9-A : qualité de crédit des hypothèques ?",            "BMO_annual_report_FY2024-0113", r"credit quality|impaired|delinquen|loan-to-value|LTV"),
    ("Q6-A : parle-t-il d'IA ?",                              "BNS_annual_report_FY2025-0112", r"\bAI\b|artificial intelligence"),
]
for label, cid, kw in CHECKS:
    print(f"\n######## {label}")
    window(cid, kw)


######## Q1-A : le total de la PCL 2025 y est-il ?

--- RBC_annual_report_FY2025-0235 ---
   …od $ 215 $ 126 $ 231 $ 572 $ 223 $ 90 $ 168 $ 481 Provision for credit losses Transfers to stage 1 157 (153) (4) – 99 (97) (2) – Transfers to stage 2 (41) 49 (8) – (23)…
   …$ 305 $ 966 $ 211 $ 1,482 $ 280 $ 793 $ 155 $ 1,228 Provision for credit losses Transfers to stage 1 594 (593) (1) – 537 (537) – – Transfers to stage 2 (96) 100 (4) – (75)…

######## Q1-C : le total de la PCL 2025 y est-il ?

--- RBC_annual_report_FY2025-0151 ---
   …f our 2025 Annual Consolidated Financial Statements. (3) Geographic information is based on residence of borrower. Provision for credit losses by portfolio and geography Table 74 For the year ended October 31 (Millions of Canadian dollars, except for percentage amounts) 2025 2024 Residential mortgages…

######## Q2-B : LCR au 31 octobre 2025 ?

--- TD_annual_report_FY2025-0182 ---
   …1 The LCR is calculated in accordance with OSFI’s LAR guideline, which is r

### pourquoi Q4, Q8 et Q10 échouent

In [11]:
ALT = {
    2:  "What was the Liquidity Coverage Ratio as at October 31, 2025?",
    4:  "What does the bank say about cybersecurity and technology risk?",
    5:  "What was the CET1 capital ratio at October 31, 2024?",
    8:  "How sensitive are net interest income and economic value of equity to interest rate changes?",
    10: "What emerging risks does the bank identify?",
}
GOLD_KW  = {2: r"liquidity coverage ratio|\bLCR\b", 4: r"cyber", 5: r"\bCET1\b", 8: r"economic value of equity|\bEVE\b", 10: r"emerging risk"}
MIN_HITS = {2: 3, 4: 3, 5: 3, 8: 2, 10: 1}

def subset_ranks(idx, q):
    s = emb[idx] @ embed_query(q)
    order = (-s).argsort()
    return {idx[i]: r + 1 for r, i in enumerate(order)}

out = []
for Q in QUERIES:
    qid = Q["id"]
    if qid not in ALT:
        continue
    sub = chunks[(chunks.bank_code == Q["bank"]) & (chunks.fiscal_year == Q["year"])]
    idx = sub.index.to_numpy()
    kre = re.compile(GOLD_KW[qid], re.I)
    hits = sub.text.map(lambda t: len(kre.findall(t)))
    gold = sub[(hits >= MIN_HITS[qid]) & (sub.part == "MD&A")].assign(hits=hits).nlargest(3, "hits")
    r_orig, r_alt = subset_ranks(idx, Q["q"]), subset_ranks(idx, ALT[qid])
    for i, g in gold.iterrows():
        m = kre.search(g.text)
        pos = len(model.tokenizer(g.text[:m.start()], add_special_tokens=False)["input_ids"])
        out.append({"Q": qid, "chunk_id": g.chunk_id, "section": g.section[:32], "occurrences": g.hits,
                    "position mot-clé (tokens)": pos, "tokens MiniLM": g.n_tokens_minilm,
                    f"rang /{len(sub)}": r_orig[i], "rang sans nom de banque": r_alt[i]})
d = pd.DataFrame(out)
print(d.to_string(index=False))
print(f"\nChunks cibles où le mot-clé apparaît APRÈS le token 512 (donc non lu par MiniLM) : {(d['position mot-clé (tokens)'] > 512).sum()} / {len(d)}")

 Q                       chunk_id                          section  occurrences  position mot-clé (tokens)  tokens MiniLM  rang /401  rang sans nom de banque  rang /358  rang /354  rang /412  rang /359
 2   TD_annual_report_FY2025-0181                    Managing Risk           10                         30            705        5.0                        3        NaN        NaN        NaN        NaN
 2   TD_annual_report_FY2025-0182                    Managing Risk            8                          2            673        2.0                        1        NaN        NaN        NaN        NaN
 4  BNS_annual_report_FY2025-0108                  Risk Management           12                        414            931        NaN                        1       17.0        NaN        NaN        NaN
 4  BNS_annual_report_FY2025-0159                  Risk Management            6                        406            829        NaN                       10       32.0        NaN        NaN  

trancher les cas incertains

In [12]:
def grep_chunk(cid, pattern, n=4, before=120, after=260):
    t = chunks.loc[chunks.chunk_id == cid].iloc[0].text
    ms = list(re.finditer(pattern, t, re.I))
    print(f"\n######## {cid} | {len(ms)} occurrence(s) de /{pattern}/")
    for m in ms[:n]:
        s = max(0, m.start() - before)
        print("   …" + " ".join(t[s:m.end() + after].split()) + "…")

grep_chunk("RBC_annual_report_FY2025-0235", r"\bTotal\b")                         # Q1-A : total de la PCL ?
grep_chunk("RBC_annual_report_FY2025-0151", r"\bTotal\b")                         # Q1-C : total de la PCL ?
grep_chunk("TD_annual_report_FY2025-0182",  r"liquidity coverage ratio|Total HQLA")   # Q2-B : valeur du LCR ?
grep_chunk("CIBC_annual_report_FY2024-0075", r"\d{1,2}\.\d\s?%")                   # Q5-B : niveau du CET1 ?
grep_chunk("RBC_annual_report_FY2024-0176", r"gross impaired loans|\bGIL\b")      # Q7-A : libellé de la ligne ?
grep_chunk("RBC_annual_report_FY2024-0102", r"\bTotal\b|gross impaired loans")    # Q7-B : chiffres du tableau ?
grep_chunk("BMO_annual_report_FY2024-0113", r"residential mortgage|loan-to-value|\bLTV\b|impaired")  # Q9-A


######## RBC_annual_report_FY2025-0235 | 2 occurrence(s) de /\bTotal\b/
   …Impaired (Millions of Canadian dollars) Stage 1 Stage 2 Stage 3 Total Stage 1 Stage 2 Stage 3 Total Residential mortgages Balance at beginning of period $ 215 $ 126 $ 231 $ 572 $ 223 $ 90 $ 168 $ 481 Provision for credit…
   …Stage 1 Stage 2 Stage 3 Total Stage 1 Stage 2 Stage 3 Total Residential mortgages Balance at beginning of period $ 215 $ 126 $ 231 $ 572 $ 223 $ 90 $ 168 $ 481 Provision for credit losses Transfers to stage 1 157 (1…

######## RBC_annual_report_FY2025-0151 | 6 occurrence(s) de /\bTotal\b/
   …1,773 1,034 Total PCL on impaired loans $ 3,734 $ 2,620 Canada (1) Residential mortgages $ 152 $ 96 Person…
   …1,086 663 Total $ 3,017 $ 2,234 U.S. (1) Retail $ 52 $ 33 Wholesale…
   …Wholesale 225 366 Total $ 277 $ 399 Other International (1) Retail $ (22) $ (19…
   …Wholesale 462 6 Total $ 440 $ (13) Total PCL on impaired loans $ 3,734 $ 2,620 Total PCL on perform…

######## TD_annual_report_FY202

In [14]:
QUERIES_V2 = {
    1:  "What was the provision for credit losses in fiscal 2025?",
    2:  "What was the Liquidity Coverage Ratio as at October 31, 2025?",
    3:  "How does the bank describe its approach to climate risk management?",
    4:  "What does the bank say about cybersecurity and technology risk?",
    5:  "What was the CET1 capital ratio at October 31, 2024?",
    7:  "What were the gross impaired loans in fiscal 2024?",
    8:  "How sensitive are net interest income and economic value of equity to interest rate changes?",
    9:  "How is the credit quality of the residential mortgage portfolio?",
    10: "What emerging risks does the bank identify for 2024?",
}   # Q6 inchangée : pas de banque dans la question

results_v2 = {}
for Q in QUERIES:
    q2 = QUERIES_V2.get(Q["id"], Q["q"])
    where = build_where(Q["bank"], Q["year"])
    for method in ("topk", "mmr"):
        results_v2[(Q["id"], method)] = search(f"{Q['id']}-v2", q2, method, where)

def kw_rate(res_dict):
    return {m: np.mean([bool(re.search(Q["kw"], r["text"], re.I)) for Q in QUERIES for r in res_dict[(Q["id"], m)][:3]])
            for m in ("topk", "mmr")}

print("Part des 3 premiers résultats qui contiennent le mot-clé attendu (indicateur automatique, PAS le precision@3) :")
print("  v1 (avec le nom de la banque) :", {k: f"{v:.0%}" for k, v in kw_rate(results).items()})
print("  v2 (sans le nom de la banque) :", {k: f"{v:.0%}" for k, v in kw_rate(results_v2).items()})
for Q in QUERIES:
    a = {r["chunk_id"] for r in results[(Q["id"], "topk")][:3]}
    b = {r["chunk_id"] for r in results_v2[(Q["id"], "topk")][:3]}
    print(f"Q{Q['id']:>2} top-k : {len(a & b)}/3 chunks identiques entre v1 et v2")

pd.DataFrame(RETRIEVAL_LOG).to_json(REPORTS / "retrieval_log.jsonl", orient="records", lines=True, force_ascii=False)

Part des 3 premiers résultats qui contiennent le mot-clé attendu (indicateur automatique, PAS le precision@3) :
  v1 (avec le nom de la banque) : {'topk': '47%', 'mmr': '40%'}
  v2 (sans le nom de la banque) : {'topk': '67%', 'mmr': '50%'}
Q 1 top-k : 2/3 chunks identiques entre v1 et v2
Q 2 top-k : 1/3 chunks identiques entre v1 et v2
Q 3 top-k : 2/3 chunks identiques entre v1 et v2
Q 4 top-k : 0/3 chunks identiques entre v1 et v2
Q 5 top-k : 2/3 chunks identiques entre v1 et v2
Q 6 top-k : 3/3 chunks identiques entre v1 et v2
Q 7 top-k : 2/3 chunks identiques entre v1 et v2
Q 8 top-k : 2/3 chunks identiques entre v1 et v2
Q 9 top-k : 2/3 chunks identiques entre v1 et v2
Q10 top-k : 0/3 chunks identiques entre v1 et v2


In [20]:
JUDGE = {
    1: {"RBC_annual_report_FY2025-0235": False, "RBC_annual_report_FY2025-0141": False,
        "RBC_annual_report_FY2025-0160": False, "RBC_annual_report_FY2025-0246": False,
        "RBC_annual_report_FY2025-0237": False, "RBC_annual_report_FY2025-0305": False,
        "RBC_annual_report_FY2025-0188": False},
    2: {"TD_annual_report_FY2025-0028": False, "TD_annual_report_FY2025-0396": False,
        "TD_annual_report_FY2025-0114": False, "TD_annual_report_FY2025-0086": False},
    3: {"BMO_annual_report_FY2025-0148": True, "BMO_annual_report_FY2025-0150": True,
        "BMO_annual_report_FY2025-0151": True, "BMO_annual_report_FY2025-0084": True,
        "BMO_annual_report_FY2025-0146": False, "BMO_annual_report_FY2025-0147": False},
    4: {"BNS_annual_report_FY2025-0357": False, "BNS_annual_report_FY2025-0007": False,
        "BNS_annual_report_FY2025-0103": False, "BNS_annual_report_FY2025-0160": False,
        "BNS_annual_report_FY2025-0108": True, "BNS_annual_report_FY2025-0112": True,
        "BNS_annual_report_FY2025-0111": False},
    5: {"CIBC_annual_report_FY2024-0350": False, "CIBC_annual_report_FY2024-0288": False,
        "CIBC_annual_report_FY2024-0073": False, "CIBC_annual_report_FY2024-0256": False},
    6: {"TD_annual_report_FY2025-0173": True, "TD_annual_report_FY2025-0016": False},
    7: {"RBC_annual_report_FY2024-0176": True, "RBC_annual_report_FY2024-0102": True,
        "RBC_annual_report_FY2024-0177": False, "RBC_annual_report_FY2024-0142": False,
        "RBC_annual_report_FY2024-0124": False, "RBC_annual_report_FY2024-0178": False,
        "RBC_annual_report_FY2024-0213": False},
    8: {"TD_annual_report_FY2024-0035": False, "TD_annual_report_FY2024-0304": False,
        "TD_annual_report_FY2024-0055": False, "TD_annual_report_FY2024-0266": False,
        "TD_annual_report_FY2024-0239": False},
    9: {"BMO_annual_report_FY2024-0108": False},
    10: {"BNS_annual_report_FY2024-0005": False, "BNS_annual_report_FY2024-0105": False,
         "BNS_annual_report_FY2024-0007": False, "BNS_annual_report_FY2024-0280": False,
         "BNS_annual_report_FY2024-0156": False, "BNS_annual_report_FY2024-0107": True,
         "BNS_annual_report_FY2024-0108": True, "BNS_annual_report_FY2024-0103": False,
         "BNS_annual_report_FY2024-0122": False},
}

def verdict(qid, cid):
    return JUDGE.get(qid, {}).get(cid)          # None = pas encore jugé

def p_at_3(res_dict, lenient=False):
    ok = (lambda v: v in (True, "partial")) if lenient else (lambda v: v is True)
    rows = []
    for Q in QUERIES:
        for method in ("topk", "mmr"):
            ids = [r["chunk_id"] for r in res_dict[(Q["id"], method)][:3]]
            rows.append({"Q": Q["id"], "méthode": method,
                         "pertinents": sum(ok(verdict(Q["id"], c)) for c in ids),
                         "non jugés": sum(verdict(Q["id"], c) is None for c in ids)})
    d = pd.DataFrame(rows); d["P@3"] = (d.pertinents / 3).round(2)
    return d

def todo(res_dict, label):
    seen = set()
    print(f"\n--- À juger ({label}) ---")
    for Q in QUERIES:
        for method in ("topk", "mmr"):
            for r in res_dict[(Q["id"], method)][:3]:
                key = (Q["id"], r["chunk_id"])
                if verdict(*key) is None and key not in seen:
                    seen.add(key)
                    print(f"Q{Q['id']} | {r['chunk_id']} | …{snippet(r['text'], Q['kw'])}…")
    print(f"{len(seen)} chunk(s) à juger")

for label, rd in [("v1 (avec le nom de la banque)", results), ("v2 (sans le nom de la banque)", results_v2)]:
    d = p_at_3(rd)
    print(f"\n=== {label} ===")
    if d["non jugés"].sum():
        todo(rd, label)
    else:
        for name, lenient in (("strict", False), ("indulgent", True)):
            print(f"Precision@3 {name} :", p_at_3(rd, lenient).groupby("méthode")["P@3"].mean().round(3).to_dict())


=== v1 (avec le nom de la banque) ===

--- À juger (v1 (avec le nom de la banque)) ---
Q1 | RBC_annual_report_FY2025-0151 | …onsolidated Financial Statements. (3) Geographic information is based on residence of borrower. Provision for credit losses by portfolio and geography Table 74 For the year ended October 31 (Millions of Canadian dollars, exc…
Q2 | TD_annual_report_FY2025-0182 | …1 The LCR is calculated in accordance with OSFI’s LAR guideline, which is reflective 2 Unweighted inflow and outflow values are outstanding balances maturing or callable of liquidity-related requirements published by the BCBS. The LCR for the quarter wi…
Q4 | BNS_annual_report_FY2025-0006 | …investment and innovation. As this space continues to undergo rapid change, oversight of cybersecurity and Strong progress the ethical use of data and artificial intelligence are key areas of focus for the Committee. We also welcomed…
Q5 | CIBC_annual_report_FY2024-0075 | …CET1 ratio The CET1 ratio at October 31, 202

In [22]:
# Mes verdicts sur les chunks restants (True = le passage contient la réponse, False = non)
t = chunks.loc[chunks.chunk_id == "RBC_annual_report_FY2025-0151", "text"].iloc[0]
verdicts = {
    (1, "RBC_annual_report_FY2025-0151"): "4,356" in t,
    (2, "TD_annual_report_FY2025-0181"): True,
    (2, "TD_annual_report_FY2025-0182"): True,
    (4, "BNS_annual_report_FY2025-0006"): False,
    (5, "CIBC_annual_report_FY2024-0075"): False,
    (6, "BNS_annual_report_FY2025-0112"): True,
    (6, "BNS_annual_report_FY2024-0110"): True,
    (9, "BMO_annual_report_FY2024-0113"): False,
    (9, "BMO_annual_report_FY2024-0114"): True,
    (9, "BMO_annual_report_FY2024-0116"): False,
}
for (q, cid), v in verdicts.items():
    JUDGE.setdefault(q, {})[cid] = v

# Calcul du score pour la version 1 (avec le nom de la banque) et la version 2 (sans)
for label, rd in [("v1 (avec le nom de la banque)", results), ("v2 (sans le nom de la banque)", results_v2)]:
    d = p_at_3(rd)
    print(f"\n=== {label} ===")
    if d["non jugés"].sum():
        todo(rd, label)
    else:
        print("Precision@3 :", p_at_3(rd).groupby("méthode")["P@3"].mean().round(3).to_dict())
        print(p_at_3(rd).pivot(index="Q", columns="méthode", values="P@3").to_string())


=== v1 (avec le nom de la banque) ===
Precision@3 : {'mmr': 0.233, 'topk': 0.333}
méthode   mmr  topk
Q                  
1        0.00  0.00
2        0.33  0.33
3        0.67  1.00
4        0.00  0.00
5        0.00  0.00
6        0.67  1.00
7        0.33  0.67
8        0.00  0.00
9        0.33  0.33
10       0.00  0.00

=== v2 (sans le nom de la banque) ===
Precision@3 : {'mmr': 0.4, 'topk': 0.434}
méthode   mmr  topk
Q                  
1        0.00  0.00
2        0.33  0.67
3        1.00  0.67
4        0.67  0.67
5        0.00  0.00
6        0.67  1.00
7        0.33  0.33
8        0.00  0.00
9        0.33  0.33
10       0.67  0.67


### Precision@3 (10 requêtes, 3 premiers résultats, jugement manuel)

| Méthode | v1 (avec le nom de la banque) | v2 (sans le nom) |
|---|---|---|
| Top-k | 0,33 (10/30) | 0,43 (13/30) |
| MMR | 0,23 (7/30) | 0,40 (12/30) |

Par type de question :
- Questions descriptives (Q3, Q4, Q6, Q9, Q10) : top-k 0,47 → 0,67 et MMR 0,33 → 0,67. Le gain vient surtout de Q4 et Q10.
- Questions à chiffre (Q1, Q2, Q5, Q7, Q8) : stable à 0,20 pour top-k et 0,13 pour MMR. Q1, Q5 et Q8 ont 0 dans les deux versions.

Critère de jugement : un chunk est pertinent s'il contient l'information qui répond à la question (le chiffre demandé, ou une description substantielle du sujet). Une mention en passant, ou la bonne banque et la bonne année sans la réponse, comptent comme non pertinent.

Limites : 30 jugements seulement ; les requêtes v2 ont été écrites après avoir vu les échecs de la v1 ; les verdicts ont été posés à partir d'extraits et doivent être relus ; deux cas limites (BMO FY2024 0114 et 0116, qualité de crédit des hypothèques).

Décision : séparer la banque et l'année (filtre de métadonnées) du texte de la recherche sémantique. Pistes pour les questions à chiffre : recherche hybride, re-ranking, extraction de tableaux.

### comparer les chunks de risque de crédit de RBC seulement contre toutes les banques

In [23]:
q = "provision for credit losses and impaired loans"      # sans nom de banque, comme en v2
cases = {
    "RBC seulement (filtre banque + catégorie)": {"$and": [{"bank_code": "RBC"}, {"risk_category": "Credit risk"}]},
    "Toutes les banques (filtre catégorie seulement)": {"risk_category": "Credit risk"},
    "Aucun filtre": None,
}
for label, where in cases.items():
    res = retrieve_topk(q, k=5, where=where)
    print(f"\n=== {label} ===")
    for r in res:
        print(f"  sim {r['sim']:.2f} | {r['bank_code']:4} {r['fiscal_year']} | {r['risk_category']:<14} | {r['citation'].split(' – ')[2][:45]}")
    print("  Banques :", sorted({r['bank_code'] for r in res}))


=== RBC seulement (filtre banque + catégorie) ===
  sim 0.62 | RBC  2024 | Credit risk    | Critical Accounting Policies And Estimates (p
  sim 0.61 | RBC  2024 | Credit risk    | Note 2 Summary of material accounting policie
  sim 0.58 | RBC  2025 | Credit risk    | Critical Accounting Policies And Estimates (p
  sim 0.57 | RBC  2024 | Credit risk    | Note 5 Loans and allowance for credit losses 
  sim 0.57 | RBC  2025 | Credit risk    | Note 2 Summary of material accounting policie
  Banques : ['RBC']

=== Toutes les banques (filtre catégorie seulement) ===
  sim 0.71 | CIBC 2025 | Credit risk    | Credit Risk (p.75-76)
  sim 0.71 | CIBC 2024 | Credit risk    | Credit Risk (p.81-82)
  sim 0.70 | BMO  2025 | Credit risk    | MD&A (p.33-34)
  sim 0.69 | BNS  2024 | Credit risk    | Management’s Discussion And Analysis Highligh
  sim 0.68 | BNS  2025 | Credit risk    | Risk Management (p.95-96)
  Banques : ['BMO', 'BNS', 'CIBC']

=== Aucun filtre ===
  sim 0.71 | CIBC 2025 | Credit ri

### Filtrage par métadonnées
Requête : « provision for credit losses and impaired loans », top-5.
- **Filtre banque (RBC) + catégorie « Credit risk » :** 5 résultats, tous RBC (2024 et 2025).
- **Filtre catégorie seul, toutes les banques :** 5 résultats venant de BMO, BNS et CIBC. Aucun de RBC ni de TD.
- **Sans filtre :** identique au cas précédent, car les 5 meilleurs sont déjà classés « Credit risk ». Le filtre de catégorie n'a donc rien changé ici.

Conclusions :
- Sans filtre de banque, une question sur RBC ne renvoie aucun passage de RBC : le filtre de métadonnées est indispensable. Au notebook 04, le nom de la banque et l'année seront extraits de la question pour construire le filtre.
- Les similarités plus élevées sans filtre (0,68 à 0,71 contre 0,57 à 0,62 pour RBC seul) ne mesurent pas la qualité : elles ne se comparent pas d'un filtre à l'autre.

Limites : une seule requête testée ; la pertinence des résultats RBC n'a pas été vérifiée à la main ; les titres de section de RBC sont approximatifs (pas de signets dans le PDF).